# 4 · Diagnose faults and stop on a lost acknowledgment

Learn to preserve current and historical fault evidence, keep unknown fault codes visible, and treat an unacknowledged write as an unknown outcome.

**Self-contained notebook · simulator default · optional human-operated hardware**

From the repository root, activate your virtual environment, install with `python -m pip install -e ".[tutorials]"`, then launch `python -m jupyterlab examples/tutorials`. Select this environment's Python kernel. Run cells top to bottom with Shift+Enter, or choose **Restart Kernel and Run All Cells**. See [setup and troubleshooting](README.md#start-here).

Every demonstration and hardware helper is defined below. The notebook imports only the controller library and Python's standard library; no tutorial script is loaded. Run cells from top to bottom.

[All four tutorials](README.md)

## 1. Command sequence

A: fixture injects codes 2 and 999 → `?L, ?F, ?FH` → fixture removes active conditions → `?L, ?F, ?FH` → close.

B: a separate standby simulator applies `P=0.2500` but drops its reply → report UNKNOWN → close. Exactly one write is attempted. No `L=1`, automatic reset, reconnect, recovery query or command replay occurs.

The controller supplies CR/LF framing and waits for each reply. Queries start with `?`; writes use `=`. No `OK` acknowledgment is assumed. Source: supplied [Verdi manual](../../verdi.manual_v5.pdf), Tables 5-1, 5-3 and 5-4; [protocol mapping](../../docs/PROTOCOL.md).

## 2. Import the controller API

Imports do not discover ports, open connections or send commands.

In [ ]:
"""Tutorial 4: preserve fault evidence and stop after a lost acknowledgment."""

from coherent_verdi import (
    ControllerConfig,
    Fault,
    Model,
    ProtocolError,
    SimulatedTransport,
    TransportError,
    VerdiController,
    VerdiError,
)

## 3. Define the application steps

Each cell defines one small function. The complete sequence is run in section 4.

### 3.1 Read active and historical faults

Read state, current faults and history. Unknown codes are preserved, not treated as healthy.

In [ ]:
def read_fault_report(laser: VerdiController) -> tuple[tuple[Fault, ...], tuple[Fault, ...]]:
    state = laser.laser_state()  # ?L
    active = laser.faults()  # ?F
    history = laser.faults(history=True)  # ?FH
    print(f"Laser state: {state.name}")
    for label, faults in (("Active", active), ("History", history)):
        print(f"{label} fault codes: {[fault.code for fault in faults]}")
        for fault in faults:
            print(f"  {fault.code}: {fault.description}; known={fault.known}")
    if active:
        print("STOP: diagnose every active code, including unknown codes. No automatic reset.")
    return active, history

### 3.2 Handle one attempted write

A transport or parsing failure means the outcome is unknown. Return False and let the caller end the session; never replay the write.

In [ ]:
def set_power_once(laser: VerdiController, target_w: float) -> bool:
    """One attempted write in an already approved state; caller closes on False."""
    try:
        laser.set_power_w(target_w)
    except (TransportError, ProtocolError) as exc:
        print(f"{type(exc).__name__}: {exc}")
        print("Outcome UNKNOWN: the setpoint may have changed. Do not retry or continue querying.")
        return False
    return True  # Acknowledged only; this alone is not state or optical verification.

## 4. Run the simulator

The fixture setup below is synthetic. The exercise target of 0.25 W and ceiling of 0.5 W are not physical safety limits.

### 4.1 Prepare the fault exercise

These fixture changes are local to a simulator. Removing active conditions does not clear its retained history or enable it.

In [ ]:
def demonstrate_faults() -> None:
    sim = SimulatedTransport(Model.V5)
    sim.set_faults(2, 999)  # External interlock plus an intentionally unknown fixture code.
    try:
        with VerdiController(sim, ControllerConfig(Model.V5)) as laser:
            read_fault_report(laser)
            sim.set_faults()  # Remove synthetic active conditions; NOT a real reset command.
            print("\nFixture conditions removed; read again without enabling:")
            read_fault_report(laser)
    except VerdiError as exc:
        print(f"STOP: incomplete fault report: {exc}. Preserve evidence; no retry.")
        raise

### 4.2 Prepare the lost-reply exercise

A separate standby simulator applies a setpoint and drops its acknowledgment. The application cannot determine the outcome from the missing reply.

In [ ]:
def demonstrate_lost_reply() -> None:
    sim = SimulatedTransport(Model.V5)
    config = ControllerConfig(Model.V5, allow_writes=True, power_limit_w=0.5)
    with VerdiController(sim, config) as laser:
        sim.inject_timeout(after_apply=True)  # Fake applies P, then drops its acknowledgment.
        if not set_power_once(laser, 0.25):
            print("Ending this session now. No reconnect, state replay or enable.")
    print(f"Simulator wire requests: {sim.requests!r}")
    print("Connection released. Communication close did not undo the attempted setpoint.")

### 4.3 Prepare the simulator demonstration

The function below owns the simulator and its controller lifetime. Each call starts a fresh exercise and leaves no worker or open session.

In [ ]:
def simulate_handle_faults() -> None:
    demonstrate_faults()
    demonstrate_lost_reply()

### 4.4 Execute the demonstration

Run this short cell to call the functions just defined. Rerun it to begin with a fresh simulator.

In [ ]:
simulate_handle_faults()

## 5. Expected simulator result

- First report: FAULT; active/history codes `[2, 999]`; code 2 is an external interlock fault; code 999 has `known=False`.
- Second report: active codes `[]`; history still `[2, 999]`; state remains FAULT (simulator policy).
- Lost reply: `ResponseTimeout`, `Outcome UNKNOWN`, then the session ends.
- Final request tuple: `(b'P=0.2500\r\n',)` — one attempted write only.

The timeout is intentionally caught. Closing communication did not undo the write. An actual serial timeout makes the session unusable; even reopening is not proof that late replies have cleared.

## 6. Try one small change

Change the fixture's `999` to another unknown positive code such as `998` and rerun. Then change `after_apply=True` to `False`: the same UNKNOWN message remains correct, because the caller cannot tell whether the command executed from a missing reply alone.

## 7. Optional real-hardware session for a human operator

Complete the [hardware review procedure](../../HARDWARE_VALIDATION.md) first. Install `python -m pip install -e ".[tutorials,serial]"` in this environment. The hardware functions below are fully visible and call the application functions in section 3 directly.

Fill in the actual model, native port and matching baud. This lesson keeps writes disabled. Keep `RUN_HARDWARE=False` for ordinary Run All.

After you enable it, **CONNECT** permits the selected connection and one `?SV` query. Verify the reported device and version. **RUN** permits the displayed lesson. Any other answer cancels. These prompts confirm intent, not site approval.

### 7.1 Import serial interfaces

These imports alone perform no I/O. `contextmanager` lets a `with` block release the connection even on an exception.

In [ ]:
from collections.abc import Iterator
from contextlib import contextmanager

from coherent_verdi import Query, SerialConfig, open_serial

### 7.2 Validate the power configuration

Validation happens before connection. Read-only sessions reject power settings; write sessions require an explicit approved ceiling and target.

In [ ]:
def hardware_power_config(
    model: Model,
    *,
    allow_writes: bool = False,
    target_w: float | None = None,
    power_limit_w: float | None = None,
    active_fault_clear_reply: str | None = None,
) -> ControllerConfig:
    """Validate explicit hardware power settings before opening a connection."""
    config = ControllerConfig(model, allow_writes, power_limit_w, active_fault_clear_reply)
    if allow_writes:
        if power_limit_w is None:
            raise ValueError("Supply the approved power_limit_w; there is no hardware default")
        if (
            isinstance(target_w, bool)
            or not isinstance(target_w, (int, float))
            or not 0 <= target_w <= config.effective_power_limit_w
            or float(f"{target_w:.4f}") > config.effective_power_limit_w
        ):
            raise ValueError("target_w must be finite, nonnegative and within the approved ceiling")
    elif target_w is not None or power_limit_w is not None:
        raise ValueError("Read-only lessons do not accept power settings")
    return config

### 7.3 Connect, identify and release

This helper prompts before opening, sends `?SV` first and closes communication when the `with` block ends. An error stops further commands. **Connection close is not physical shutdown**; use the operator's abort procedure when state is uncertain.

In [ ]:
@contextmanager
def hardware_connection(
    serial_config: SerialConfig,
    config: ControllerConfig,
) -> Iterator[VerdiController | None]:
    """Ask CONNECT, read ?SV first, and release the connection on leaving the block."""
    print(f"HARDWARE: {config.model}, port={serial_config.port}, baud={serial_config.baudrate}")
    print(f"Timeout: {serial_config.timeout_s} s. First interaction: ?SV only.")
    if input("With candidate/connection approval recorded, type CONNECT: ").strip() != "CONNECT":
        print("Cancelled before opening the port.")
        yield None
        return
    try:
        with VerdiController(open_serial(serial_config, hardware_allowed=True), config) as laser:
            print(f"Reported software: {laser.query(Query.SOFTWARE)}")
            yield laser
    except BaseException:
        print("STOP: session failed/interrupted. No reconnect, retries or blind cleanup writes.")
        print("Physical state may be unknown. Use the operator's approved abort procedure.")
        raise
    finally:
        print("Communication scope ended; releasing a connection does not shut down the laser.")

### 7.4 Enter the operator's settings

Nothing connects when you run this configuration cell. Set `RUN_HARDWARE=True` only for an attended, approved session. Restore False afterwards.

The manual does not specify the no-active-fault reply for `?F`. Keep `ACTIVE_FAULT_CLEAR_REPLY=None` until Stage 1 records its exact meaning for this firmware. Then enter the verified text here; otherwise a clear-looking response stops the physical session. `SYSTEM OK` is documented for `?FH` only. Simulator defaults use their explicit fixture convention.

In [ ]:
RUN_HARDWARE = False  # Change only for an approved, attended physical session.
HARDWARE_PORT = None  # Set to the operator-confirmed native port, e.g. "COM3".
HARDWARE_MODEL = None  # Set to Model.V2, Model.V5 or Model.V6 after checking the label.
HARDWARE_BAUDRATE = None  # Set to the actual front-panel baud rate.
HARDWARE_TIMEOUT_S = 1.0  # Adjust to the validated transaction deadline.
ACTIVE_FAULT_CLEAR_REPLY = None  # Set only after Stage 1 verifies the exact ?F clear text.

### 7.5 Run the visible hardware sequence

Calls `read_fault_report` from section 3. There is no fixture injection/removal, timeout demonstration, power write or reset.

The call completes in one cell; no connection waits between cells. A new run prompts again.

In [ ]:
if RUN_HARDWARE:
    serial_config = SerialConfig(
        HARDWARE_PORT, baudrate=HARDWARE_BAUDRATE, timeout_s=HARDWARE_TIMEOUT_S
    )
    config = hardware_power_config(
        HARDWARE_MODEL, active_fault_clear_reply=ACTIVE_FAULT_CLEAR_REPLY
    )
    print("Plan: read state, active faults and history; no writes.")
    with hardware_connection(serial_config, config) as laser:
        if laser is not None:
            if input("Verify device/version and approved scope; type RUN: ").strip() == "RUN":
                read_fault_report(laser)
            else:
                print("Cancelled after ?SV; no lesson commands sent.")
else:
    print("Hardware section skipped. Set explicit operator configuration to use it.")

## 8. What this establishes

Simulator runs verify software behavior, not physical response or calibration. A status sample is sequential, not an interlock or proof of a safe beam path. Review the [simulator assumptions](../../docs/SIMULATOR.md) and [operator guide](README.md#human-operated-hardware) before physical use.